# Filter top Reddit posts per subreddit

## Assumptions
- Input parquet files live in `data/raw/reddit/pushshift_1`.
- Output keeps only the columns used by the requested rules plus `rank_in_subreddit`.
- `created_utc` must already be present as a string in exact `YYYY-MM-DD` format; rows outside that format are dropped.
- `author kiểu bot` is interpreted here as usernames containing `bot` (case-insensitive) or equal to `AutoModerator`.

## Plan
1. Check each parquet file for the required columns and log skipped files.
2. Clean rows, drop missing values, remove bot-like authors, validate `created_utc`, and apply quality thresholds.
3. Keep the top `1000` rows per subreddit by `score`, add `rank_in_subreddit`, and write one parquet file.


In [ ]:
# Import các thư viện tối thiểu cần cho việc đọc, lọc và ghi parquet.
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Khai báo input, output và số lượng bài tối đa cần giữ cho mỗi subreddit.
INPUT_DIR = Path("data/raw/reddit/pushshift_1")
OUTPUT_PATH = Path("data/processed/top_posts_per_subreddit.parquet")
TOP_K_PER_SUBREDDIT = 1000

# Đây là các cột bắt buộc phải có trước khi xử lý dữ liệu.
required_columns = [
    "subreddit",
    "author",
    "title",
    "score",
    "num_comments",
    "upvote_ratio",
    "created_utc",
]

# Starter thresholds. Adjust these if you want a stricter or looser quality bar.
min_score = 10
min_comments = 3
min_upvote_ratio = 0.80
min_title_chars = 15

# Các rule hỗ trợ việc bỏ giá trị trống và loại tác giả dạng bot.
BOT_AUTHOR_KEYWORD = "bot"
MISSING_TEXT_VALUES = {"", "none", "null", "nan"}
INVALID_AUTHOR_VALUES = MISSING_TEXT_VALUES | {"[deleted]", "[removed]", "deleted", "removed"}

INPUT_DIR, OUTPUT_PATH


In [ ]:
from IPython.display import display


# Chỉ đọc các cột cần thiết, đồng thời bỏ qua file nào thiếu schema bắt buộc.
def read_required_columns(parquet_path: Path) -> tuple[pd.DataFrame | None, list[str]]:
    schema_names = set(pq.ParquetFile(parquet_path).schema.names)
    missing_columns = [column for column in required_columns if column not in schema_names]
    if missing_columns:
        return None, missing_columns

    df = pd.read_parquet(parquet_path, columns=required_columns)
    return df, []


# Làm sạch từng row rồi áp toàn bộ rule chất lượng trong một chỗ duy nhất.
def clean_and_filter_rows(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    text_columns = ["subreddit", "author", "title", "created_utc"]
    numeric_columns = ["score", "num_comments", "upvote_ratio"]

    # Chuẩn hóa text trước, sau đó đổi các giá trị rỗng phổ biến thành NA.
    for column in text_columns:
        df[column] = df[column].astype("string").str.strip()
        df.loc[df[column].str.lower().isin(MISSING_TEXT_VALUES), column] = pd.NA

    # Ép các cột số về numeric để lọc ngưỡng an toàn hơn.
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # Bỏ ngay các row thiếu dữ liệu ở những cột bắt buộc.
    df = df.dropna(subset=required_columns).copy()

    author_lower = df["author"].str.lower()
    title_length = df["title"].str.len()
    valid_created_utc = df["created_utc"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
    parsed_created_utc = pd.to_datetime(df["created_utc"], format="%Y-%m-%d", errors="coerce")

    # Filter cuối cùng: bỏ author không hợp lệ, author kiểu bot, ngày sai format và row dưới ngưỡng chất lượng.
    df = df[
        ~author_lower.isin(INVALID_AUTHOR_VALUES)
        & ~author_lower.eq("automoderator")
        & ~author_lower.str.contains(BOT_AUTHOR_KEYWORD, regex=False, na=False)
        & valid_created_utc.fillna(False)
        & parsed_created_utc.notna()
        & (df["score"] >= min_score)
        & (df["num_comments"] >= min_comments)
        & (df["upvote_ratio"] >= min_upvote_ratio)
        & (title_length >= min_title_chars)
    ].copy()

    df["created_utc"] = parsed_created_utc.loc[df.index].dt.strftime("%Y-%m-%d")

    return df


# Sắp xếp theo chất lượng bài viết rồi chỉ giữ top K trong từng subreddit.
def take_top_posts(df: pd.DataFrame, top_k: int) -> pd.DataFrame:
    sort_columns = ["subreddit", "score", "num_comments", "upvote_ratio", "title"]
    ascending = [True, False, False, False, True]
    ranked = df.sort_values(sort_columns, ascending=ascending, kind="stable")
    return ranked.groupby("subreddit", sort=False, group_keys=False).head(top_k).reset_index(drop=True)


In [ ]:
# Lấy danh sách toàn bộ file parquet đầu vào.
parquet_paths = sorted(INPUT_DIR.glob("*.parquet"))
if not parquet_paths:
    raise FileNotFoundError(f"No parquet files found under {INPUT_DIR}")

# `stats` dùng để theo dõi nhanh số file và số row qua từng bước xử lý.
stats = {
    "total_files": len(parquet_paths),
    "processed_files": 0,
    "skipped_files_missing_columns": 0,
    "input_rows": 0,
    "rows_after_filters": 0,
}
skipped_files = []
best_by_subreddit = {}

# Duyệt từng file: kiểm tra schema, làm sạch row, rồi cập nhật top bài cho từng subreddit.
for parquet_path in parquet_paths:
    df, missing_columns = read_required_columns(parquet_path)
    if missing_columns:
        skipped_files.append(
            {
                "file": parquet_path.name,
                "missing_columns": ", ".join(missing_columns),
            }
        )
        stats["skipped_files_missing_columns"] += 1
        continue

    stats["processed_files"] += 1
    stats["input_rows"] += len(df)

    cleaned = clean_and_filter_rows(df)
    stats["rows_after_filters"] += len(cleaned)
    if cleaned.empty:
        continue

    local_top = take_top_posts(cleaned, TOP_K_PER_SUBREDDIT)

    # Gộp top mới từ file hiện tại với top đã tích lũy trước đó của cùng subreddit.
    for subreddit, group in local_top.groupby("subreddit", sort=False):
        previous_top = best_by_subreddit.get(subreddit)
        if previous_top is None:
            best_by_subreddit[subreddit] = group.copy()
            continue

        merged_top = pd.concat([previous_top, group], ignore_index=True)
        best_by_subreddit[subreddit] = take_top_posts(merged_top, TOP_K_PER_SUBREDDIT)

if not best_by_subreddit:
    raise ValueError("No rows left after cleaning and quality filters.")

# Gộp kết quả cuối, đánh số hạng trong từng subreddit và chỉ giữ các cột đầu ra cần thiết.
result = pd.concat(best_by_subreddit.values(), ignore_index=True)
result = take_top_posts(result, TOP_K_PER_SUBREDDIT)
result["rank_in_subreddit"] = result.groupby("subreddit", sort=False).cumcount() + 1
result = result[
    [
        "subreddit",
        "rank_in_subreddit",
        "author",
        "title",
        "score",
        "num_comments",
        "upvote_ratio",
        "created_utc",
    ]
].reset_index(drop=True)

# Tạo thư mục nếu chưa có rồi ghi toàn bộ kết quả ra một file parquet.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
result.to_parquet(OUTPUT_PATH, index=False)

# In summary để kiểm tra nhanh quy mô dữ liệu sau khi lọc.
summary = pd.DataFrame(
    [
        {
            **stats,
            "subreddits_in_output": result["subreddit"].nunique(),
            "output_rows": len(result),
            "output_path": str(OUTPUT_PATH),
        }
    ]
)

display(summary)
if skipped_files:
    display(pd.DataFrame(skipped_files).head(20))

result.head(10)


In [ ]:
# Verify rằng rank trong mỗi subreddit bắt đầu từ 1, liên tục và không vượt quá top K.
rank_check = result.groupby("subreddit", sort=False)["rank_in_subreddit"].agg(["min", "max", "count"])
rank_check = rank_check.rename(columns={"count": "rows"})

assert rank_check["min"].eq(1).all()
assert (rank_check["max"] == rank_check["rows"]).all()
assert rank_check["rows"].le(TOP_K_PER_SUBREDDIT).all()

display(rank_check.head(10))
result.head(10)
